STEP 1

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# read counts data
expr = pd.read_csv("SRP_counts.tsv", sep="\t", index_col="Gene")
meta = pd.read_csv("metadata.tsv", sep="\t")

assert list(expr.columns) == meta.refinebio_accession_code.tolist()
print("genes x samples:", expr.shape)

genes x samples: (43363, 335)


In [5]:
meta[["region", "dx"]] = meta.refinebio_subject.str.split("_", n=1, expand=True)
meta["donor"] = meta.refinebio_title.str.split("_").str[0]

print(pd.crosstab(meta.dx, meta.region))
print("unique donors:", meta.donor.nunique())

region            ancg  dlpfc  nacc
dx                                 
bipolar disorder    25     28    30
control             28     28    27
major depression    27     30    32
schizophrenia       27     28    25
unique donors: 96


In [6]:
run_to_lib = meta.set_index("refinebio_accession_code").refinebio_title

print("duplicate runs:", run_to_lib.duplicated().sum())

expr_lib = expr.T.groupby(run_to_lib).mean().T

meta_lib = (
    meta.drop_duplicates("refinebio_title")
    .set_index("refinebio_title")
    .loc[expr_lib.columns]
)

print("genes x libraries:", expr_lib.shape)

duplicate runs: 69
genes x libraries: (43363, 266)


In [7]:
print("range before log:", expr_lib.values.min(), expr_lib.values.max())

log_expr = np.log2(expr_lib + 1)

range before log: -0.1583617006285569 197.4761356701856


STEP 2

In [8]:
# selecting variable genes
gene_variance = log_expr.var(axis=1)
top_5000 = gene_variance.nlargest(5000).index
expr_5000 = log_expr.loc[top_5000].T
print(expr_5000.shape)

(266, 5000)


In [21]:
ancg_samples = meta_lib.index[meta_lib["region"] == "ancg"]

log_expr_ancg = log_expr[ancg_samples]

print(log_expr_ancg.shape)

(43363, 89)


In [22]:
gene_variance_ancg = log_expr_ancg.var(axis=1)

top_5000_ancg = gene_variance_ancg.nlargest(5000).index

expr_5000_ancg = log_expr_ancg.loc[top_5000_ancg].T

print(expr_5000_ancg.shape)

(89, 5000)


In [35]:
# Keep only AnCg libraries
expr_AnCg = expr_5000[
    expr_5000.index.str.contains("_AnCg_")
].copy()

print("AnCg expression matrix shape:", expr_AnCg.shape)

AnCg expression matrix shape: (89, 5000)


In [9]:
!pip install scikit-learn-extra

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 17.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-learn-extra: filename=scikit_learn_extra-0.3.0-cp313-cp313-linux_x86_64.whl size=2154569 sha256=f04196b74ec7c655a7ba5814cfa4467705554748730198628bd0f979bca1d11b
  Stored in directory: /root/.cache/pip/wheels/17/4b/b8/6b6711681d0981b110c9cc91ad6d1ebd88adf1547e1da301fc
Successfully built scikit-learn-extra


In [10]:
from sklearn_extra.cluster import KMedoids

In [36]:
# PAM with k=2
pam_2 = KMedoids(
    n_clusters=2,
    random_state=42,
    metric="euclidean"
)

clusters_2 = pam_2.fit_predict(expr_AnCg)

print("Cluster counts:")
print(pd.Series(clusters_2).value_counts().sort_index())

Cluster counts:
0    55
1    34
Name: count, dtype: int64


In [37]:
k_values = [2, 3, 4, 5, 6]

pam_results = {}

for k in k_values:
    pam = KMedoids(
        n_clusters=k,
        random_state=42,
        metric="euclidean"
    )

    pam_results[k] = pam.fit_predict(expr_AnCg)

    print(f"k = {k}")
    print(pd.Series(pam_results[k]).value_counts().sort_index())
    print()

k = 2
0    55
1    34
Name: count, dtype: int64

k = 3
0    34
1    27
2    28
Name: count, dtype: int64

k = 4
0    36
1    11
2    27
3    15
Name: count, dtype: int64

k = 5
0    29
1    10
2    21
3    13
4    16
Name: count, dtype: int64

k = 6
0    28
1    10
2    21
3    15
4     9
5     6
Name: count, dtype: int64



In [38]:
gene_numbers = [10, 100, 1000, 5000, 10000]

gene_sets = {}

for n in gene_numbers:
    top_genes = gene_variance.nlargest(n).index
    gene_sets[n] = log_expr.loc[top_genes].T

    print(n, gene_sets[n].shape)

10 (266, 10)
100 (266, 100)
1000 (266, 1000)
5000 (266, 5000)
10000 (266, 10000)


In [39]:
comparison_2_3 = pd.crosstab(
    pam_results[2],
    pam_results[3]
)

print(comparison_2_3)

col_0   0   1   2
row_0            
0      28  27   0
1       6   0  28


In [40]:
def compare_clusters(k1, k2):
    return pd.crosstab(
        pam_results[k1],
        pam_results[k2]
    )

print("k = 2 vs k = 3")
print(compare_clusters(2, 3))

print("\nk = 3 vs k = 4")
print(compare_clusters(3, 4))

print("\nk = 4 vs k = 5")
print(compare_clusters(4, 5))

print("\nk = 5 vs k = 6")
print(compare_clusters(5, 6))

k = 2 vs k = 3
col_0   0   1   2
row_0            
0      28  27   0
1       6   0  28

k = 3 vs k = 4
col_0   0  1   2   3
row_0               
0      17  1   3  13
1      19  8   0   0
2       0  2  24   2

k = 4 vs k = 5
col_0   0  1   2   3  4
row_0                  
0      23  4   2   0  7
1       4  4   0   0  3
2       2  2  19   0  4
3       0  0   0  13  2

k = 5 vs k = 6
col_0   0   1   2   3  4  5
row_0                      
0      23   0   0   0  4  2
1       0  10   0   0  0  0
2       0   0  20   0  0  1
3       0   0   0  13  0  0
4       5   0   1   2  5  3


In [43]:
meta_AnCg = meta_lib.loc[expr_AnCg.index].copy()

for k in range(2, 6):
    pam_diagnosis = pd.DataFrame({
        "library": expr_AnCg.index,
        "PAM_cluster": pam_results[k],
        "diagnosis": meta_AnCg["dx"].values
    })

    print(f"\nPAM k={k} cluster vs. diagnosis:")
    print(pd.crosstab(
        pam_diagnosis["PAM_cluster"],
        pam_diagnosis["diagnosis"]
    ))


PAM k=2 cluster vs. diagnosis:
diagnosis    bipolar disorder  control  major depression  schizophrenia
PAM_cluster                                                            
0                          12       14                14             15
1                          10        8                 8              8

PAM k=3 cluster vs. diagnosis:
diagnosis    bipolar disorder  control  major depression  schizophrenia
PAM_cluster                                                            
0                          10        7                 5             12
1                           4        8                10              5
2                           8        7                 7              6

PAM k=4 cluster vs. diagnosis:
diagnosis    bipolar disorder  control  major depression  schizophrenia
PAM_cluster                                                            
0                           7       11                 9              9
1                           1        3  

In [44]:
from sklearn.metrics import silhouette_score
from scipy.stats import chi2_contingency
import pandas as pd

summary = []

for k in range(2, 6):
    clusters = pam_results[k]

    cluster_sizes = (
        pd.Series(clusters)
        .value_counts()
        .sort_index()
        .tolist()
    )
    cluster_sizes_str = " / ".join(map(str, cluster_sizes))

    sil = silhouette_score(expr_AnCg, clusters, metric="euclidean")

    contingency = pd.crosstab(
        clusters,
        meta_AnCg["dx"].values
    )
    chi2, p_value, dof, expected = chi2_contingency(contingency)

    summary.append({
        "k": k,
        "Cluster sizes": cluster_sizes_str,
        "Silhouette score": round(sil, 3),
        "Chi squared p value": round(p_value, 3)
    })

summary_df = pd.DataFrame(summary)

print(summary_df.to_string(index=False))

 k          Cluster sizes  Silhouette score  Chi squared p value
 2                55 / 34             0.044                0.881
 3           34 / 27 / 28             0.012                0.318
 4      36 / 11 / 27 / 15             0.052                0.001
 5 29 / 10 / 21 / 13 / 16             0.072                0.086


In [51]:
gene_counts = [10, 100, 1000, 10000]

# Restrict every gene set to the 89 AnCg libraries
gene_sets_AnCg = {}

for n in gene_counts:
    gene_sets_AnCg[n] = gene_sets[n].loc[expr_AnCg.index]
    print(n, gene_sets_AnCg[n].shape)

10 (89, 10)
100 (89, 100)
1000 (89, 1000)
10000 (89, 10000)


In [52]:
pam_gene_results = {}

for n in gene_counts:
    pam_gene_results[n] = {}

    for k in k_values:
        pam = KMedoids(
            n_clusters=k,
            random_state=42,
            metric="euclidean"
        )

        pam_gene_results[n][k] = pam.fit_predict(gene_sets_AnCg[n])

        print(f"{n} genes, k = {k}")
        print(
            pd.Series(pam_gene_results[n][k])
            .value_counts()
            .sort_index()
        )
        print()

10 genes, k = 2
0    40
1    49
Name: count, dtype: int64

10 genes, k = 3
0    40
1    30
2    19
Name: count, dtype: int64

10 genes, k = 4
0    39
1    14
2    17
3    19
Name: count, dtype: int64

10 genes, k = 5
0    39
1    14
2    13
3    15
4     8
Name: count, dtype: int64

10 genes, k = 6
0    30
1    14
2    13
3    14
4     8
5    10
Name: count, dtype: int64

100 genes, k = 2
0    48
1    41
Name: count, dtype: int64

100 genes, k = 3
0    44
1    21
2    24
Name: count, dtype: int64

100 genes, k = 4
0    26
1    21
2    24
3    18
Name: count, dtype: int64

100 genes, k = 5
0    15
1    20
2    22
3    13
4    19
Name: count, dtype: int64

100 genes, k = 6
0    12
1    18
2    21
3    12
4    11
5    15
Name: count, dtype: int64

1000 genes, k = 2
0    60
1    29
Name: count, dtype: int64

1000 genes, k = 3
0    39
1    24
2    26
Name: count, dtype: int64

1000 genes, k = 4
0    28
1    14
2    12
3    35
Name: count, dtype: int64

1000 genes, k = 5
0    26
1    14
2   

In [53]:
from scipy.stats import chi2_contingency
import itertools
import pandas as pd

k = 2
results = []

for n1, n2 in itertools.combinations(gene_counts, 2):

    clusters1 = pam_gene_results[n1][k]
    clusters2 = pam_gene_results[n2][k]

    contingency = pd.crosstab(clusters1, clusters2)

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    results.append({
        "Comparison: number of genes": f"{n1} vs {n2}",
        "Chi-square": round(chi2, 2),
        "DF": dof,
        "p-value": p_value,
        "Minimum expected count": round(expected.min(), 2)
    })

gene_comparison_df = pd.DataFrame(results)

print(gene_comparison_df.to_string(index=False))

Comparison: number of genes  Chi-square  DF      p-value  Minimum expected count
                  10 vs 100       40.72   1 1.754761e-10                   18.43
                 10 vs 1000        1.33   1 2.493359e-01                   13.03
                10 vs 10000        2.70   1 1.000771e-01                   15.73
                100 vs 1000        7.76   1 5.334504e-03                   13.36
               100 vs 10000        2.49   1 1.146706e-01                   16.12
              1000 vs 10000        0.00   1 9.647299e-01                   11.40


STEP 4

In [55]:
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
import pandas as pd

results = []

for k in range(2, 6):

    clusters = pam_results[k]

    contingency = pd.crosstab(
        clusters,
        meta_AnCg["dx"].values
    )

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    results.append({
        "Clustering method": f"5000 genes (k={k})",
        "Chi-square": chi2,
        "Degrees of freedom": dof,
        "p-value": p_value,
        "Number of cluster": k,
        "Minimum expected count": expected.min()
    })


for n in [10, 100, 1000, 10000]:

    clusters = pam_gene_results[n][2]

    contingency = pd.crosstab(
        clusters,
        meta_AnCg["dx"].values
    )

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    results.append({
        "Clustering method": f"{n} genes (k=2)",
        "Chi-square": chi2,
        "Degrees of freedom": dof,
        "p-value": p_value,
        "Number of cluster": 2,
        "Minimum expected count": expected.min()
    })


results_df = pd.DataFrame(results)

In [56]:
reject, adjusted_p, _, _ = multipletests(
    results_df["p-value"],
    method="fdr_bh"
)

results_df["Adjusted p-value"] = adjusted_p

In [57]:
results_df = results_df[
    [
        "Clustering method",
        "Chi-square",
        "Degrees of freedom",
        "p-value",
        "Adjusted p-value",
        "Number of cluster",
        "Minimum expected count"
    ]
]

results_df["Chi-square"] = results_df["Chi-square"].round(2)
results_df["p-value"] = results_df["p-value"].round(4)
results_df["Adjusted p-value"] = results_df["Adjusted p-value"].round(4)
results_df["Minimum expected count"] = results_df["Minimum expected count"].round(2)

print(results_df.to_string(index=False))

Clustering method  Chi-square  Degrees of freedom  p-value  Adjusted p-value  Number of cluster  Minimum expected count
 5000 genes (k=2)        0.67                   3   0.8809            0.8851                  2                    8.40
 5000 genes (k=3)        7.03                   6   0.3185            0.5197                  3                    6.67
 5000 genes (k=4)       26.91                   9   0.0014            0.0116                  4                    2.72
 5000 genes (k=5)       19.11                  12   0.0860            0.2940                  5                    2.47
   10 genes (k=2)        1.78                   3   0.6184            0.8246                  2                    9.89
  100 genes (k=2)        6.03                   3   0.1102            0.2940                  2                   10.13
 1000 genes (k=2)        3.47                   3   0.3248            0.5197                  2                    7.17
10000 genes (k=2)        0.65           